# Custom L-BFGS Tutorial

This tutorial demonstrates `ls_bayesian`'s metric-generic L-BFGS backend,
`CustomLBFGSOptimizer` (`ls_bayesian.optimization.algorithms.custom_lbfgs`), together with
its two composable components: a step-size strategy
(`ls_bayesian.optimization.components.line_search`) and a correction-pair acceptance
strategy (`ls_bayesian.optimization.components.cautious_update`). Unlike `ScipyLBFGSBOptimizer`
(see the companion `scipy_lbfgs` tutorial), which is hardcoded to the Euclidean inner
product, `CustomLBFGSOptimizer` performs its two-loop recursion, line search, and
correction-pair bookkeeping entirely in whatever inner product
`OptimizationModel.evaluate_inner_product` defines -- e.g. a Cameron-Martin inner product
induced by a Bayesian prior's covariance. We demonstrate the effect of the inner product
directly, by minimizing the same quadratic objective under two different metrics and
comparing convergence.

## Components

`CustomLBFGSOptimizer` is assembled from three independent pieces, each a separate degree
of freedom:

- [`LineSearchStrategy`][ls_bayesian.optimization.components.line_search.LineSearchStrategy]:
  selects a step size along the search direction. We use
  [`ArmijoBacktrackingLineSearch`][ls_bayesian.optimization.components.line_search.ArmijoBacktrackingLineSearch],
  backtracking on the Armijo sufficient-decrease condition.
- [`CorrectionPairAcceptanceStrategy`][ls_bayesian.optimization.components.cautious_update.CorrectionPairAcceptanceStrategy]:
  decides whether a new $(s, y)$ correction pair is stored in the limited-memory history.
  We use
  [`CautiousUpdateStrategy`][ls_bayesian.optimization.components.cautious_update.CautiousUpdateStrategy],
  the strengthened curvature condition of Li & Fukushima (2001), which keeps the
  implicit inverse-Hessian approximation positive definite; the trivial
  [`AlwaysAcceptStrategy`][ls_bayesian.optimization.components.cautious_update.AlwaysAcceptStrategy]
  is also available.
- [`CustomLBFGSSettings`][ls_bayesian.optimization.algorithms.custom_lbfgs.CustomLBFGSSettings]:
  memory size, iteration budget, and the gradient-norm convergence tolerance.

All three are evaluated in the inner product `model.evaluate_inner_product` defines, so
the same components work unchanged regardless of the geometry.

In [ ]:
from typing import override

import matplotlib.pyplot as plt
import numpy as np

from ls_bayesian.common.logging import BaseLogger, LoggerSettings
from ls_bayesian.optimization.algorithms.custom_lbfgs import (
    CustomLBFGSOptimizer,
    CustomLBFGSSettings,
)
from ls_bayesian.optimization.components.cautious_update import (
    CautiousUpdateSettings,
    CautiousUpdateStrategy,
)
from ls_bayesian.optimization.components.line_search import (
    ArmijoBacktrackingLineSearch,
    ArmijoBacktrackingLineSearchSettings,
)
from ls_bayesian.optimization.model import OptimizationModel

rng = np.random.default_rng(0)
PARAMETER_DIM = 5

## Objective: a weighted quadratic bowl

We minimize the convex quadratic $f(m) = \frac{1}{2}(m - m^*)^T A (m - m^*)$, with a
random SPD matrix $A$ and known minimizer $m^*$. `evaluate_gradient` must return the
**Riesz representer of the objective's derivative under `evaluate_inner_product`**
(see `CustomLBFGSOptimizer`'s docstring), not necessarily the Euclidean gradient: with
inner-product weight $W$, the representer of the Euclidean
gradient $A(m - m^*)$ is $W^{-1}A(m - m^*)$. `inner_product_matrix` defaults to the
identity, reproducing the standard Euclidean setting.

In [ ]:
class WeightedQuadraticModel(OptimizationModel):
    r"""Quadratic bowl f(m) = 1/2 (m-m*)^T A (m-m*), with the gradient/Hessian-vector
    product returned as the Riesz representer under a (possibly weighted) inner product.
    Weight defaults to the identity, i.e. the standard Euclidean inner product."""

    def __init__(
        self,
        matrix: np.ndarray,
        minimizer: np.ndarray,
        inner_product_matrix: np.ndarray | None = None,
    ) -> None:
        self.matrix = matrix
        self.minimizer = minimizer
        self.inner_product_matrix = (
            np.eye(matrix.shape[0]) if inner_product_matrix is None else inner_product_matrix
        )

    @override
    def evaluate_cost(self, parameter_vector: np.ndarray) -> float:
        difference = parameter_vector - self.minimizer
        return float(0.5 * difference @ self.matrix @ difference)

    @override
    def evaluate_gradient(self, parameter_vector: np.ndarray) -> np.ndarray:
        euclidean_gradient = self.matrix @ (parameter_vector - self.minimizer)
        return np.linalg.solve(self.inner_product_matrix, euclidean_gradient)

    @override
    def evaluate_hessian_vector_product(
        self, parameter_vector: np.ndarray, direction_vector: np.ndarray
    ) -> np.ndarray:
        return np.linalg.solve(self.inner_product_matrix, self.matrix @ direction_vector)

    @override
    def evaluate_inner_product(self, first_vector: np.ndarray, second_vector: np.ndarray) -> float:
        return float(first_vector @ self.inner_product_matrix @ second_vector)


factor = rng.random((PARAMETER_DIM, PARAMETER_DIM))
matrix = factor @ factor.T + PARAMETER_DIM * np.eye(PARAMETER_DIM)  # random SPD
minimizer = rng.standard_normal(PARAMETER_DIM)
initial_guess = np.zeros(PARAMETER_DIM)

## Assembling the optimizer

We build one `CustomLBFGSOptimizer` with default component settings, and reuse it for
both runs below -- only the model's inner product changes between them.

In [ ]:
line_search = ArmijoBacktrackingLineSearch(ArmijoBacktrackingLineSearchSettings())
acceptance_strategy = CautiousUpdateStrategy(CautiousUpdateSettings())
settings = CustomLBFGSSettings(gradient_norm_tolerance=1e-10, maximum_num_iterations=200)
logger = BaseLogger(LoggerSettings(), prefix="custom-lbfgs")

optimizer = CustomLBFGSOptimizer(settings, line_search, acceptance_strategy, logger=logger)

## Run 1: the standard Euclidean inner product

With `inner_product_matrix=None`, the model uses the standard Euclidean inner product.
L-BFGS has to build up an approximation of $A$'s curvature from scratch, one correction
pair per iteration.

In [ ]:
euclidean_model = WeightedQuadraticModel(matrix, minimizer)
result_euclidean = optimizer.run(initial_guess, euclidean_model)

print(f"iterations: {result_euclidean.num_iterations}")
print(f"distance to minimizer: {np.linalg.norm(result_euclidean.result - minimizer):.3e}")

## Run 2: the inner product matched to the Hessian

Now we set `inner_product_matrix=matrix`, i.e. the inner product **is** the objective's
(constant) Hessian $A$ -- the role a Cameron-Martin inner product induced by a Gaussian
prior's covariance plays for a real posterior. In this metric, the gradient's representer
is exactly $A^{-1}A(m - m^*) = m - m^*$, so the two-loop recursion's identity seed already
gives the exact Newton direction $p = -(m - m^*)$, and a full step $\tau=1$ lands exactly
on $m^*$. We therefore expect this run to converge in a single iteration, versus the
many needed above.

In [ ]:
natural_model = WeightedQuadraticModel(matrix, minimizer, inner_product_matrix=matrix)
result_natural = optimizer.run(initial_guess, natural_model)

print(f"iterations: {result_natural.num_iterations}")
print(f"distance to minimizer: {np.linalg.norm(result_natural.result - minimizer):.3e}")

## Comparing convergence

Both runs converge to the same minimizer, confirming `CustomLBFGSOptimizer` is correct
regardless of the inner product -- but the metric matched to the problem's curvature
converges dramatically faster, exactly the payoff a well-chosen Cameron-Martin inner
product gives for a real Bayesian inverse problem.

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(result_euclidean.loss_history, label="Euclidean inner product", marker="o")
ax.semilogy(result_natural.loss_history, label="Hessian-matched inner product", marker="o")
ax.set_xlabel("iteration")
ax.set_ylabel("loss (log scale)")
ax.set_title("CustomLBFGSOptimizer convergence under two inner products")
ax.legend()
plt.show()

In [ ]:
np.testing.assert_allclose(result_euclidean.result, minimizer, atol=1e-4)
np.testing.assert_allclose(result_natural.result, minimizer, atol=1e-4)
assert result_natural.num_iterations < result_euclidean.num_iterations
print("Both runs converged to the known minimizer; the matched metric needed far fewer iterations.")